# i2sb MRI recon: plain sampler vs CG data consistency, step by step

One val slice from a trained i2sb-TGVN run, reconstructed two ways from its bridge start:

1. **i2sb:** the plain reverse bridge.
2. **i2sb-MRI:** the same, with the CG data-consistency step on every x̂₀ (I2SB-Recon), on the
   same bridge noise.

Every `SHOW_EVERY` network evaluations it shows the state x_t and x̂₀ (before and after CG for
i2sb-MRI), plus how much energy each carries outside the support. The `INIT` toggle picks the
starting point, and with it the run trained for that start. Everything is the training code:
`training/i2sb_recon.py::_measure`, `sb/i2sb_recon.py::i2sb_recon_sample`.

In [ ]:
%matplotlib inline
import os, pathlib, sys
import numpy as np
import torch
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd()
if not (ROOT / "physics").is_dir():
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from datasets.registry import build_loader
from sb.i2sb_recon import bridge_noise_scale, i2sb_recon_sample
from scripts.eval_i2sb_recon_dc import load_run
from training.forward_op import fixed_val_subset
from training.i2sb_recon import _measure, measurement_support
from training.metrics import compute_metrics

try:                                   # the INIT toggle; a plain string without ipywidgets
    import ipywidgets as widgets
    INIT = widgets.ToggleButtons(options=["adjoint", "acs", "guide"], value="adjoint",
                                 description="INIT")
    display(INIT)
except ImportError:
    INIT = "adjoint"

In [ ]:
# ===================== SETTINGS -- pick INIT above, then run the cells below =====================
RUNS = "trained_nets/guided_recon/knee/s3/sbtgvn_c8_{start}_R8_g1_tau0.09"  # one run per start
CKPT = "net_bestval.ckpt"     # or "net.ckpt": the latest validation (sampled from its EMA)
X1_OVERRIDE = None         # None: the run's own start. Else sample from another start than it
                           # was trained for (a mismatch -- diagnostics only)
SLICE = 0                  # index into the run's fixed val subset
NFE = None                 # None: the run's val_nfe
SHOW_EVERY = 5             # intermediates every N evaluations
DC_ITERS, DC_LAM = 7, 7e-5 # the CG step of the i2sb-MRI sampler (the configs' values)
SUPPORT_THRESH = 0.9       # hard support from the operator's online ESPIRiT eigenvalues; None = off
SUPPORT_STATE = False      # also mask the bridge state after every step
VAL_NOISE = 0.0            # ADDED measurement noise std: 0 = the scan as acquired (its own noise
                           # only, the realistic case); None = the run's val_noise_std
SEED = 0

start = INIT if isinstance(INIT, str) else INIT.value
run = load_run(RUNS.format(start=start), CKPT)
net, bridge, i2, tr, dev = (run[k] for k in ("net", "bridge", "i2", "tr", "device"))
meas = dict(run["meas"], x1_source=X1_OVERRIDE or run["meas"]["x1_source"])
nfe = NFE or run["nfe"]
vcfg = dict(run["cfg"]["data"]["val"], batch_size=1)
loader = fixed_val_subset(build_loader(vcfg, shuffle=False, drop_last=False),
                          tr.get("val_slices"), run["seed"])
batch = loader.dataset[SLICE]
batch = [t[None] if torch.is_tensor(t) else torch.as_tensor(t)[None] for t in batch]
gen = torch.Generator(device=dev).manual_seed(SEED)
x0, x1, y, E, sigma_n, organ, gt, guide = _measure(
    batch, net, dev, kspace_type=run["kspace_type"],
    noise_std=run["noise_std"] if VAL_NOISE is None else VAL_NOISE,
    generator=gen, guide_offset=0, **meas)
support = None if SUPPORT_THRESH is None else measurement_support(y, E, meas, SUPPORT_THRESH)
kappa = bridge_noise_scale(x1, i2.get("tau_scale", "none"), i2.get("tau_ref"))
print(f"run {RUNS.format(start=start)} | x1={meas['x1_source']} tau={i2['tau']} nfe={nfe} | "
      f"sigma_n={float(sigma_n.reshape(-1)[0]):.4f} | support "
      + ("off" if support is None else f"{float(support.float().mean()):.1%} of the FOV"))

In [ ]:
def sample(dc, sup, trace=None):
    devs = [dev] if dev.type == "cuda" else []
    with torch.random.fork_rng(devices=devs):
        torch.manual_seed(SEED)                     # the same bridge noise for every variant
        return i2sb_recon_sample(net, y, E, bridge, sigma_n=sigma_n, x1=x1, nfe=nfe,
                                 deterministic=bool(i2.get("deterministic", False)),
                                 posterior=i2.get("posterior", "ddpm"), verbose=False,
                                 guide=guide, noise_scale=kappa, dc_iters=dc, dc_lam=DC_LAM,
                                 support=sup, support_state=SUPPORT_STATE, trace=trace)[0]

traces = {"i2sb": [], "i2sb-MRI": []}
recons = {"i2sb": sample(0, support, traces["i2sb"]),
          "i2sb-MRI": sample(DC_ITERS, support, traces["i2sb-MRI"])}
if support is not None:                             # the same two without the support
    recons["i2sb, no support"] = sample(0, None)
    recons["i2sb-MRI, no support"] = sample(DC_ITERS, None)

mr = tr.get("metric_range", 1.0)
rows = {"start x1": x1, **recons}
for k, v in rows.items():
    m = compute_metrics(gt.abs(), v.abs(), data_range=mr)
    print(f"{k:<24} PSNR {float(m['psnr']):6.2f}  SSIM {float(m['ssim']):.4f}  "
          f"NRMSE {float(m['nrmse']):.4f}")

### Intermediates

Rows show the state x_t and x̂₀; for i2sb-MRI also x̂₀ after CG and CG's own change; with a
support, x̂₀ after it (what the posterior step uses). All magnitudes share one grey scale (the
target's 99th percentile).

In [ ]:
vmax = float(gt.abs().flatten().quantile(0.99))
for name, tr_ in traces.items():
    picks = tr_[::SHOW_EVERY] + ([tr_[-1]] if (len(tr_) - 1) % SHOW_EVERY else [])
    keys = [("x_t", "x_t"), ("x0_pre", "x0 pred")]
    if name == "i2sb-MRI":
        keys += [("x0_dc", "x0 after CG"), ("delta", "|CG change| (x5)")]
    if support is not None:
        keys += [("x0_post", "x0 after support")]
    fig, ax = plt.subplots(len(keys), len(picks), figsize=(2.6 * len(picks), 3.4 * len(keys)),
                           squeeze=False)
    for j, t in enumerate(picks):
        for i, (k, lab) in enumerate(keys):
            img = (t["x0_dc"] - t["x0_pre"]).abs() if k == "delta" else t[k].abs()
            ax[i, j].imshow(img[0, 0], cmap="magma" if k == "delta" else "gray", vmin=0,
                            vmax=0.2 * vmax if k == "delta" else vmax)
            ax[i, j].set_title(f"{lab}, step {t['step']}", fontsize=8)
            ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
    fig.suptitle(name); plt.tight_layout(); plt.show()

### Background: energy outside the support, per evaluation

RMS magnitude outside the support (or outside the target's own nonzero region when the support is
off), for the state and the predictions. A rising curve is the background diverging.

In [ ]:
outside = (~support[0, 0].cpu()) if support is not None else (gt[0, 0].abs().cpu() == 0)
rms = lambda t: float(t[0, 0][outside].abs().pow(2).mean().sqrt()) if outside.any() else 0.0
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
for a, (name, tr_) in zip(ax, traces.items()):
    steps = [t["step"] for t in tr_]
    a.plot(steps, [rms(t["x_t"]) for t in tr_], "-o", ms=3, label="state x_t")
    a.plot(steps, [rms(t["x0_pre"]) for t in tr_], "-o", ms=3, label="x0 pred")
    if name == "i2sb-MRI":
        a.plot(steps, [rms(t["x0_dc"]) for t in tr_], "-o", ms=3, label="x0 after CG")
    a.invert_xaxis(); a.set_xlabel("bridge step (t = 1 -> 0)"); a.set_title(name); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
shows = [("target", gt), ("start x1", x1)] + list(recons.items())
fig, ax = plt.subplots(2, len(shows), figsize=(3.2 * len(shows), 7.2))
for j, (k, v) in enumerate(shows):
    v = v.detach().cpu()[0, 0]
    ax[0, j].imshow(v.abs(), cmap="gray", vmin=0, vmax=vmax); ax[0, j].set_title(k, fontsize=9)
    err = (v.abs() - gt.detach().cpu()[0, 0].abs()).abs()
    ax[1, j].imshow(err, cmap="magma", vmin=0, vmax=0.2 * vmax); ax[1, j].set_title("|error| x5", fontsize=9)
for a in ax.flat:
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()